In [ ]:
import torch
import os
import numpy as np
from time import time as t
import scipy.io as sio

class Nodes(torch.nn.Module):
    def __init__(self, n=None, shape=None, traces=False, traces_additive=False,
                 tc_trace=20.0, tc_trace_neg=20.0, trace_scale=1.0, sum_input=False):
        super().__init__()
        self.n = n
        self.shape = (n,) if shape is None else shape
        self.batch_size = 1
        self.traces = traces
        self.traces_additive = traces_additive
        self.register_buffer("tc_trace", torch.tensor(tc_trace, dtype=torch.float))
        self.register_buffer("tc_trace_neg", torch.tensor(tc_trace_neg, dtype=torch.float))
        self.trace_scale = trace_scale
        self.sum_input = sum_input
        if traces:
            self.register_buffer("x", torch.zeros(1, *self.shape))
            self.register_buffer("x_neg", torch.zeros(1, *self.shape))
        else:
            self.x = None
            self.x_neg = None
        self.register_buffer("s", torch.zeros(1, *self.shape, dtype=torch.bool))

    def forward(self, x: torch.Tensor) -> None:
        if self.traces:
            decay_factor_pre = torch.exp(-1.0 / self.tc_trace)
            decay_factor_post = torch.exp(-1.0 / self.tc_trace_neg)
            self.x = self.x * decay_factor_pre + self.trace_scale * self.s.float()
            self.x_neg = self.x_neg * decay_factor_post + self.trace_scale * self.s.float()

    def reset_(self) -> None:
        self.s.zero_()
        if self.traces:
            self.x.zero_()
            self.x_neg.zero_()

    def set_batch_size(self, batch_size: int) -> None:
        self.batch_size = batch_size

    def compute_decays(self, dt: float) -> None:
        pass

class Input(Nodes):
    def __init__(self, n=None, shape=None, traces=False, traces_additive=False,
                 tc_trace=20.0, tc_trace_neg=20.0, trace_scale=1.0, sum_input=False,
                 thresh=-52.0, rest=-65.0, reset=-65.0, refrac=60, **kwargs):
        super().__init__(n, shape, traces, traces_additive, tc_trace, tc_trace_neg, trace_scale, sum_input)
        self.register_buffer("rest", torch.tensor(rest, dtype=torch.float))
        self.register_buffer("reset", torch.tensor(reset, dtype=torch.float))
        self.register_buffer("thresh", torch.tensor(thresh, dtype=torch.float))
        self.register_buffer("refrac", torch.tensor(refrac))
        self.register_buffer("refrac_count", torch.zeros(1, n))

    def forward(self, x: torch.Tensor) -> None:
        if x.dim() == 1:
            x = x.unsqueeze(0)
        batch_size, n_neurons = x.shape
        max_val, neuron_idx = torch.max(x[0], 0)
        self.s = torch.zeros_like(x, dtype=torch.bool)
        self.refrac_count = (self.refrac_count > 0).float() * (self.refrac_count - 1.0)
        if max_val > 0:
            batch_idx = 0
            if self.refrac_count[batch_idx, neuron_idx] == 0:
                self.s[batch_idx, neuron_idx] = True
                self.refrac_count[batch_idx, neuron_idx] = self.refrac
        super().forward(x)

    def reset_(self) -> None:
        super().reset_()
        self.refrac_count.zero_()

    def set_batch_size(self, batch_size: int) -> None:
        super().set_batch_size(batch_size)
        self.refrac_count = torch.zeros((batch_size, self.n), device=self.refrac_count.device)

class LIFNodes(Nodes):
    def __init__(self, n=None, shape=None, traces=False, traces_additive=False,
                 tc_trace=20.0, tc_trace_neg=20.0, trace_scale=1.0, sum_input=False,
                 thresh=-52.0, rest=-65.0, reset=-65.0, refrac=5, tc_decay=150.0,
                 dt=1.0, lbound=None, enable_astrocyte=False, alpha=None, k=None, **kwargs):
        super().__init__(n, shape, traces, traces_additive, tc_trace, tc_trace_neg, trace_scale, sum_input)
        self.dt = dt
        self.enable_astrocyte = enable_astrocyte
        self.register_buffer("rest", torch.tensor(rest, dtype=torch.float))
        self.register_buffer("reset", torch.tensor(reset, dtype=torch.float))
        self.register_buffer("thresh", torch.tensor(thresh, dtype=torch.float))
        self.register_buffer("refrac", torch.tensor(refrac))
        self.register_buffer("tc_decay", torch.tensor(tc_decay, dtype=torch.float))
        self.register_buffer("decay", torch.zeros(*self.shape))
        self.register_buffer("v", torch.zeros(1, *self.shape))
        self.register_buffer("refrac_count", torch.zeros(1, *self.shape))
        self.lbound = lbound
        self.register_buffer("G", torch.zeros(n))
        self.register_buffer("Ca", torch.zeros(n))
        self.Ca[:] = 0
        self.k = k
        self.alpha = alpha
        self.initial_thresh = thresh
        self.G_thr = 0.2
        self.Ca_duration = 1000 * 100
        self.prev_layer_s = None

    def astrocyte_input(self):
        response = torch.ones_like(self.thresh)
        if self.prev_layer_s is not None:
            if self.prev_layer_s.dim() > 1:
                prev_s_flat = self.prev_layer_s.view(-1, self.n).float().mean(dim=0)
            else:
                prev_s_flat = self.prev_layer_s.float()
            self.G = self.G - self.dt * (self.alpha * self.G - self.k * prev_s_flat)
        activated_neurons = self.G > self.G_thr
        self.Ca[activated_neurons] = self.Ca_duration
        self.Ca = torch.clamp(self.Ca - 1, min=0)
        imp_astro = torch.zeros_like(self.thresh)
        active_effect = self.Ca > 0
        imp_astro[active_effect] = 5
        response = response / (1 + imp_astro)
        return response

    def forward(self, x: torch.Tensor, current_position: int = None, adjacent_positions: list = None) -> None:
        if x.dim() == 1:
            x = x.unsqueeze(0)
        batch_size, n_neurons = x.shape
        if self.enable_astrocyte:
            self.thresh = self.initial_thresh * self.astrocyte_input()
        active_neurons = set()
        if current_position is not None:
            active_neurons.add(current_position)
        if adjacent_positions is not None:
            active_neurons.update(adjacent_positions)
        if not active_neurons:
            active_neurons = set(range(n_neurons))
        active_mask = torch.zeros(n_neurons, dtype=torch.bool, device=self.v.device)
        for idx in active_neurons:
            active_mask[idx] = True
        self.s = torch.zeros_like(x, dtype=torch.bool)
        self.v[:, active_mask] = self.decay * (self.v[:, active_mask] - self.rest) + self.rest
        self.refrac_count[:, active_mask] = (self.refrac_count[:, active_mask] > 0).float() * (self.refrac_count[:, active_mask] - self.dt)
        for neuron_idx in active_neurons:
            refrac_mask = self.refrac_count[:, neuron_idx] == 0
            self.v[:, neuron_idx] += refrac_mask.float() * x[:, neuron_idx]
            spike_mask = self.v[:, neuron_idx] >= self.thresh[neuron_idx]
            self.s[:, neuron_idx] = spike_mask
            if spike_mask.any():
                self.refrac_count[:, neuron_idx][spike_mask] = self.refrac
                self.v[:, neuron_idx][spike_mask] = self.reset
        if self.lbound is not None:
            self.v[:, active_mask] = torch.where(self.v[:, active_mask] < self.lbound, self.lbound, self.v[:, active_mask])
        super().forward(x)

    def reset_(self) -> None:
        super().reset_()
        self.v.fill_(self.rest)
        self.refrac_count.zero_()
        self.G.zero_()

    def compute_decays(self, dt) -> None:
        self.decay = torch.exp(-dt / self.tc_decay)

    def set_batch_size(self, batch_size) -> None:
        super().set_batch_size(batch_size=batch_size)
        self.v = self.rest * torch.ones(batch_size, *self.shape, device=self.v.device)
        self.refrac_count = torch.zeros(batch_size, *self.shape, device=self.refrac_count.device)
        if self.traces:
            self.x = torch.zeros(batch_size, *self.shape, device=self.x.device)
            self.x_neg = torch.zeros(batch_size, *self.shape, device=self.x_neg.device)
        self.s = torch.zeros(batch_size, *self.shape, dtype=torch.bool, device=self.s.device)

class LearningRule:
    def __init__(self, connection, nu=None, reduction=None, weight_decay=0.0, **kwargs):
        self.connection = connection
        self.nu = nu if nu is not None else [0.0, 0.0]
        self.reduction = reduction if reduction is not None else torch.mean
        self.weight_decay = weight_decay

    def update(self, **kwargs):
        if self.weight_decay != 0:
            self.connection.w.data *= (1 - self.weight_decay)

class WeightDependentPostPre(LearningRule):
    def __init__(self, connection, nu=None, reduction=None, weight_decay=0.0,
                 post_spike_weight_decay=0.0, tc_trace=20, tc_trace_neg=20, **kwargs):
        super().__init__(connection, nu, reduction, weight_decay, **kwargs)
        self.post_spike_weight_decay = post_spike_weight_decay
        self.tc_trace = tc_trace
        self.tc_trace_neg = tc_trace_neg
        self.interval = 100
        try:
            with open("STDP.txt", 'r') as fl:
                self.STDP_base = torch.zeros([101, 120])
                i = 0
                for line in fl:
                    k = 0
                    for sym in line.split():
                        self.STDP_base[i][k] = float(sym)
                        k += 1
                    i += 1
        except FileNotFoundError:
            self.STDP_base = torch.randn(101, 120) * 0.1
        self.wmin = getattr(connection, 'wmin', 0.001)
        self.wmax = getattr(connection, 'wmax', 1.0)

    def delta_w_custom_single(self, weight, delta_val):
        first_index = int(round(float(weight / self.nu[0] * 100)))
        if torch.isinf(delta_val) or torch.isnan(delta_val):
            delta_val = torch.tensor(0.0)
        second_index = int(float(delta_val) + 60)
        if second_index > 120 or second_index < 0:
            second_index = 0
        if first_index < 0:
            first_index = -first_index
        if first_index > 100:
            first_index = 100
        return self.STDP_base[first_index][second_index]

    def update(self, current_position=None, adjacent_positions=None, **kwargs):
        batch_size = self.connection.source.batch_size
        if current_position is not None and adjacent_positions is not None:
            update = torch.zeros_like(self.connection.w)
            source_s_current = self.connection.source.s[:, current_position].unsqueeze(1).unsqueeze(2).float()
            source_x_current = self.connection.source.x[:, current_position].unsqueeze(1).unsqueeze(2)
            for post_idx in adjacent_positions:
                target_s_adj = self.connection.target.s[:, post_idx].unsqueeze(1).unsqueeze(1).float()
                target_x_adj = self.connection.target.x_neg[:, post_idx].unsqueeze(1).unsqueeze(1)
                outer_product_pre = self.reduction(torch.bmm(source_s_current, target_x_adj), dim=0)
                outer_product_pre = torch.clamp(outer_product_pre, min=1e-10)
                delta_pre = self.tc_trace_neg * torch.log(outer_product_pre)
                update_pre_val = self.nu[0] * self.delta_w_custom_single(
                    self.connection.w[current_position, post_idx], delta_pre[0, 0])
                update[current_position, post_idx] += update_pre_val
                outer_product_post = self.reduction(torch.bmm(source_x_current, target_s_adj), dim=0)
                outer_product_post = torch.clamp(outer_product_post, min=1e-10)
                delta_post = -self.tc_trace * torch.log(outer_product_post)
                update_post_val = self.nu[1] * self.delta_w_custom_single(
                    self.connection.w[current_position, post_idx], delta_post[0, 0])
                update[current_position, post_idx] += update_post_val
                decay_factor = self.reduction(torch.bmm(torch.ones_like(source_x_current), target_s_adj), dim=0)
                update[current_position, post_idx] += (-self.post_spike_weight_decay) * self.connection.w[current_position, post_idx] * decay_factor[0, 0]
            self.connection.w += update
        else:
            source_s = self.connection.source.s.view(batch_size, -1).unsqueeze(2).float()
            source_x = self.connection.source.x.view(batch_size, -1).unsqueeze(2)
            target_s = self.connection.target.s.view(batch_size, -1).unsqueeze(1).float()
            target_x = self.connection.target.x_neg.view(batch_size, -1).unsqueeze(1)
            update = 0
            outer_product = self.reduction(torch.bmm(source_s, target_x), dim=0)
            outer_product = torch.clamp(outer_product, min=1e-10)
            update += self.nu[0] * self.delta_w_custom(self.tc_trace_neg * torch.log(outer_product))
            outer_product = self.reduction(torch.bmm(source_x, target_s), dim=0)
            outer_product = torch.clamp(outer_product, min=1e-10)
            update += self.nu[1] * self.delta_w_custom(-self.tc_trace * torch.log(outer_product))
            update += (-self.post_spike_weight_decay) * self.connection.w * self.reduction(
                torch.bmm(torch.ones(source_x.shape), target_s), dim=0)
            self.connection.w += update
        super().update()

class NoOp(LearningRule):
    def update(self, **kwargs):
        super().update()

class Connection(torch.nn.Module):
    def __init__(self, source, target, impulse_amplitude=0.5, impulse_amplitude_2=0.5,
                 impulse_length=40, impulse_shape_factor=0.9, invert=False,
                 update_rule=NoOp, w=None, nu=None, wmin=0, wmax=1,
                 weight_decay=0, post_spike_weight_decay=0, **kwargs):
        super().__init__()
        self.source = source
        self.target = target
        self.wmin = wmin
        self.wmax = wmax
        if w is None:
            if self.wmin == -np.inf or self.wmax == np.inf:
                w = torch.clamp(torch.rand(source.n, target.n), self.wmin, self.wmax)
            else:
                w = self.wmin + torch.rand(source.n, target.n) * (self.wmax - self.wmin)
        else:
            if self.wmin != -np.inf or self.wmax != np.inf:
                w = torch.clamp(w, self.wmin, self.wmax)
        self.w = torch.nn.Parameter(w, False)
        self.update_rule = update_rule(self, nu=nu, weight_decay=weight_decay,
                                     post_spike_weight_decay=post_spike_weight_decay)
        self.impulse_amplitude = impulse_amplitude
        self.impulse_amplitude_2 = impulse_amplitude_2
        self.impulse_length = impulse_length
        self.impulse_shape_factor = impulse_shape_factor
        self.invert = invert
        self.register_buffer("a_pre", torch.zeros(source.n))
        self.register_buffer("impulse_state", torch.zeros(source.n))

    def impulse_curve(self):
        k = self.impulse_shape_factor
        if self.invert:
            impulse_value_2 = self.impulse_amplitude / (self.impulse_length * k - 1)
            impulse_value_1 = self.impulse_amplitude / (self.impulse_length * (1 - k))
            impulse_bias = 2 * self.impulse_amplitude * (self.impulse_state > (self.impulse_length * (1 - k) + 0.5)).float() * (self.impulse_state <= (self.impulse_length * (1 - k) + 1.5)).float()
            impulse = (-impulse_value_1) * (self.impulse_state > 0).float() * (self.impulse_state <= (self.impulse_length * (1 - k) + 0.5)).float() + (-impulse_value_2) * (self.impulse_state > (self.impulse_length * (1 - k) + 1.5)).float() + impulse_bias
            return impulse
        else:
            impulse_value_2 = self.impulse_amplitude / (self.impulse_length * k - 1)
            impulse_value_1 = self.impulse_amplitude_2 / (self.impulse_length * (1 - k))
            impulse_bias = (self.impulse_amplitude + self.impulse_amplitude_2) * (self.impulse_state >= (self.impulse_length * k)).float() * (self.impulse_state < (self.impulse_length * k + 1)).float()
            impulse = (impulse_value_1) * (self.impulse_state > (self.impulse_length * k)).float() + (impulse_value_2) * (self.impulse_state > 0).float() * (self.impulse_state < (self.impulse_length * k)).float() - impulse_bias
            return impulse

    def update_impulse_state(self, s):
        self.impulse_state += (self.impulse_state > 0).float()
        s_modified = s.clone()
        if len(s_modified.shape) == 1:
            s_modified = s_modified.unsqueeze(0)
        s_modified[:, self.impulse_state > 0] = 0
        self.impulse_state += (self.impulse_state == 0).float() * s_modified.float().view(-1)
        impulse = self.impulse_curve()
        self.impulse_state *= (self.impulse_state < self.impulse_length).float()
        return impulse

    def compute(self, s: torch.Tensor) -> torch.Tensor:
        impulse = self.update_impulse_state(s)
        self.a_pre += impulse
        self.a_pre *= (self.impulse_state > 0).float()
        a_post = self.a_pre @ self.w
        return a_post.view(1, *self.target.shape)

    def update(self, **kwargs):
        self.update_rule.update(**kwargs)

    def reset_(self):
        self.a_pre.zero_()
        self.impulse_state.zero_()

class NetworkMonitor:
    def __init__(self, network, state_vars=('v', 's', 'w')):
        self.network = network
        self.state_vars = state_vars
        self.recording = {}
        self.reset_()

    def record(self):
        for var in self.state_vars:
            for name, layer in self.network.layers.items():
                if hasattr(layer, var):
                    if name not in self.recording:
                        self.recording[name] = {}
                    if var not in self.recording[name]:
                        self.recording[name][var] = []
                    data = getattr(layer, var)
                    if var == 's':
                        data = data.float()
                    self.recording[name][var].append(data.detach().clone())
            for conn_key, conn in self.network.connections.items():
                if hasattr(conn, var):
                    if conn_key not in self.recording:
                        self.recording[conn_key] = {}
                    if var not in self.recording[conn_key]:
                        self.recording[conn_key][var] = []
                    data = getattr(conn, var)
                    self.recording[conn_key][var].append(data.detach().clone())

    def get(self):
        result = {}
        for key, val_dict in self.recording.items():
            result[key] = {}
            for var, data_list in val_dict.items():
                if data_list:
                    result[key][var] = torch.stack(data_list)
        return result

    def reset_(self):
        self.recording = {}

class Network(torch.nn.Module):
    def __init__(self, dt=1.0, batch_size=1, learning=True):
        super().__init__()
        self.dt = dt
        self.batch_size = batch_size
        self.learning = learning
        self.layers = torch.nn.ModuleDict()
        self.connections = torch.nn.ModuleDict()
        self.monitors = {}

    def add_layer(self, layer, name):
        self.layers[name] = layer
        if hasattr(layer, 'compute_decays'):
            layer.compute_decays(self.dt)
        layer.set_batch_size(self.batch_size)

    def add_connection(self, connection, source, target):
        key = f"{source}_{target}"
        self.connections[key] = connection

    def add_monitor(self, monitor, name):
        self.monitors[name] = monitor

    def _get_inputs(self, layers=None):
        inpts = {}
        if layers is None:
            layers = self.layers.keys()
        for layer_name in layers:
            if layer_name not in inpts:
                layer = self.layers[layer_name]
                inpts[layer_name] = torch.zeros(self.batch_size, *layer.shape)
        for conn_key, connection in self.connections.items():
            parts = conn_key.split('_')
            if len(parts) >= 2:
                src = parts[0]
                tgt = parts[1]
                if tgt in layers:
                    source_output = connection.compute(self.layers[src].s)
                    if source_output.dim() == 1:
                        source_output = source_output.unsqueeze(0)
                    inpts[tgt] += source_output
        return inpts

    def run(self, inpts, time, injects_v=None, current_position=None, adjacent_positions=None, conn_XY=None, **kwargs):
        timesteps = int(time / self.dt)
        injects_v = injects_v or {}
        for t_step in range(timesteps):
            current_inpts = self._get_inputs()
            for layer_name, input_data in inpts.items():
                if layer_name in current_inpts:
                    if len(input_data.shape) == 3:
                        current_inpts[layer_name] += input_data[t_step]
                    else:
                        current_inpts[layer_name] += input_data
            for name, layer in self.layers.items():
                if name in current_inpts:
                    if name in injects_v:
                        inject_voltage = injects_v[name]
                        if len(inject_voltage.shape) == 1:
                            layer.v += inject_voltage
                        else:
                            layer.v += inject_voltage[t_step]
                    if name in ['Y', 'I']:
                        layer.forward(current_inpts[name], current_position=current_position, adjacent_positions=adjacent_positions)
                    else:
                        layer.forward(current_inpts[name])
            for connection in self.connections.values():
                connection.target.prev_layer_s = connection.source.s
                if connection == conn_XY:
                    connection.update(current_position=current_position,
                                     adjacent_positions=adjacent_positions,
                                     learning=self.learning, **kwargs)
                else:
                    connection.update(learning=self.learning, **kwargs)
            for monitor in self.monitors.values():
                monitor.record()

    def reset_(self):
        for layer in self.layers.values():
            layer.reset_()
        for connection in self.connections.values():
            connection.reset_()
        for monitor in self.monitors.values():
            monitor.reset_()

def bernoulli_loader(data, time=None, dt=1.0, **kwargs):
    for i in range(len(data)):
        yield data[i]

def create_adjacency_matrix(N):
    total_cells = N * N
    adjacency_matrix = np.zeros((total_cells, total_cells), dtype=int)
    for i in range(total_cells):
        row = i // N
        col = i % N
        if col > 0:
            j = i - 1
            adjacency_matrix[i][j] = 1
            adjacency_matrix[j][i] = 1
        if col < N - 1:
            j = i + 1
            adjacency_matrix[i][j] = 1
            adjacency_matrix[j][i] = 1
        if row > 0:
            j = i - N
            adjacency_matrix[i][j] = 1
            adjacency_matrix[j][i] = 1
        if row < N - 1:
            j = i + N
            adjacency_matrix[i][j] = 1
            adjacency_matrix[j][i] = 1
    return adjacency_matrix

def setup_and_run_simulation(NA, weights_mask_XY, weights_init_XY, weights_init_XI, n_steps, current_position, goal,
                            learning_rate, wmin, wmax, weight_decay, post_spike_weight_decay, reset, refrac, thresh,
                            intensity, time_steps, dt, enable_astrocyte, alpha, k):
    network = Network(dt=dt)
    input_layer = Input(n=NA, traces=True, thresh=thresh, rest=reset, reset=reset, refrac=refrac)
    output_layer = LIFNodes(n=NA, traces=True, thresh=thresh*torch.ones(NA), rest=reset, reset=reset, refrac=refrac)
    inhibitor_layer = LIFNodes(n=NA, traces=True, thresh=thresh*torch.ones(NA), rest=reset, reset=reset, refrac=refrac, dt=dt,
                              enable_astrocyte=enable_astrocyte, alpha=alpha, k=k)
    conn_XY = Connection(input_layer, output_layer, impulse_amplitude=0.5, impulse_amplitude_2=0.5,
                        impulse_length=40, impulse_shape_factor=0.9, invert=True,
                        update_rule=WeightDependentPostPre, w=weights_init_XY, nu=[10, 10],
                        wmin=wmin, wmax=wmax, weight_decay=weight_decay, post_spike_weight_decay=post_spike_weight_decay)
    conn_XI = Connection(input_layer, inhibitor_layer, impulse_amplitude=0.5, impulse_amplitude_2=0.5,
                        impulse_length=40, impulse_shape_factor=0.9, invert=True, update_rule=NoOp,
                        w=weights_init_XI, nu=[learning_rate, learning_rate], wmin=-100, wmax=wmax,
                        weight_decay=0, post_spike_weight_decay=post_spike_weight_decay)
    conn_IY = Connection(inhibitor_layer, output_layer, impulse_amplitude=0.5, impulse_amplitude_2=0.5,
                        impulse_length=40, impulse_shape_factor=0.9, invert=True, update_rule=NoOp,
                        w=-weights_init_XI, nu=[learning_rate, learning_rate], wmin=-100, wmax=wmax,
                        weight_decay=0, post_spike_weight_decay=post_spike_weight_decay)
    network.add_layer(input_layer, 'X')
    network.add_layer(output_layer, 'Y')
    network.add_layer(inhibitor_layer, 'I')
    network.add_connection(conn_XY, 'X', 'Y')
    network.add_connection(conn_XI, 'X', 'I')
    network.add_connection(conn_IY, 'I', 'Y')
    state_vars = ('v', 's', 'w', 'G', 'Ca') if enable_astrocyte else ('v', 's', 'w')
    GlobalMonitor = NetworkMonitor(network, state_vars=state_vars)
    network.add_monitor(GlobalMonitor, 'Network')
    start = t()
    positions = []
    for i in range(n_steps):
        positions.append(current_position)
        if current_position == goal:
            print("success")
            break
        adjacent_positions = []
        for j in range(NA):
            if weights_mask_XY[current_position, j] > 0 and j != current_position:
                adjacent_positions.append(j)
        weights_before = network.connections['X_Y'].w.detach().clone()
        input_data = torch.zeros(1, NA)
        input_data[0, current_position] = intensity
        sample = next(bernoulli_loader(data=(input_data/time_steps)*dt, time=time_steps)).float()
        inpts = {'X': sample}
        injects_v = {'I': torch.full((NA,), 0.02)}
        network.run(inpts=inpts, time=time_steps, injects_v=injects_v,
                   current_position=current_position, adjacent_positions=adjacent_positions,
                   conn_XY=conn_XY)
        recordings = network.monitors['Network'].get()
        spikes = np.asarray(recordings['Y']['s'])
        summed = np.squeeze(np.sum(spikes, axis=0))
        summed = summed + weights_mask_XY[current_position, :]
        max_val = np.max(summed)
        candidates = np.where((summed == max_val) & (weights_mask_XY[current_position, :] == 1))[0]
        candidates = [c for c in candidates if c != current_position]
        new_position = np.random.choice(candidates)
        delta = network.connections['X_Y'].w.detach().clone() - weights_before
        modulated_delta = delta.clone()
        for adj in adjacent_positions:
            if adj == new_position:
                coef = 1.5
            else:
                coef = 1.0
            modulated_delta[current_position, adj] = delta[current_position, adj] * coef
        network.connections['X_Y'].w.data = weights_before + modulated_delta
        network.connections['X_Y'].w.data *= torch.Tensor(weights_mask_XY).float()
        weights_2d = network.connections['X_Y'].w.detach().cpu().numpy()
        current_position = new_position
        network.reset_()
    return positions, weights_2d

def run_navigation_experiment(experiment_num):
    results_dir = "navigation_results"
    os.makedirs(results_dir, exist_ok=True)
    N = 3
    NA = N*N
    weights_mask_XY = create_adjacency_matrix(N)
    n_steps = 101
    current_position = 4
    goal = 6
    learning_rate = 1
    wmin = 0.001
    wmax = 1
    weight_decay = 0
    post_spike_weight_decay = 0.005
    reset = 0
    refrac = 40
    thresh = 7
    intensity = 15.0
    time_steps = 1000
    dt = 1
    enable_astrocyte = True
    alpha = 0.001
    k = 0.2
    weights_rand_dist_XY = np.random.normal(0.65, 0.1, size=(NA, NA))
    weights_rand_dist_XY[weights_rand_dist_XY > 0.8] = 0.8
    weights_rand_dist_XY[weights_rand_dist_XY < 0.5] = 0.5
    weights_init_XY = torch.Tensor(weights_mask_XY * weights_rand_dist_XY).float()
    weights_init_XI = torch.eye(NA)
    all_routes = []
    all_weight_matrices = []
    route_lengths = []
    QAZ_initial = weights_rand_dist_XY * weights_mask_XY
    all_weight_matrices.append(QAZ_initial.copy())
    print(f"\n{'='*60}")
    print(f"INITIAL RUN WITHOUT ASTROCYTES")
    print(f"{'='*60}")
    positions, weights_2d = setup_and_run_simulation(
        NA=NA, weights_mask_XY=weights_mask_XY, weights_init_XY=weights_init_XY, weights_init_XI=weights_init_XI,
        n_steps=n_steps, current_position=current_position, goal=goal, learning_rate=learning_rate,
        wmin=wmin, wmax=wmax, weight_decay=weight_decay, post_spike_weight_decay=post_spike_weight_decay,
        reset=reset, refrac=refrac, thresh=thresh, intensity=intensity, time_steps=time_steps, dt=dt,
        enable_astrocyte=False, alpha=alpha, k=k)
    all_routes.append(f"Route without astrocytes (initial): {positions}")
    route_lengths.append(len(positions)-1)
    for iq in range(20):
        print(f"\n{'='*60}")
        print(f"RUN WITH ASTROCYTES {iq+1} OUT OF 20")
        print(f"{'='*60}")
        positions_astro, weights_2d_astro = setup_and_run_simulation(
            NA=NA, weights_mask_XY=weights_mask_XY, weights_init_XY=weights_init_XY, weights_init_XI=weights_init_XI,
            n_steps=n_steps, current_position=current_position, goal=goal, learning_rate=learning_rate,
            wmin=wmin, wmax=wmax, weight_decay=weight_decay, post_spike_weight_decay=post_spike_weight_decay,
            reset=reset, refrac=refrac, thresh=thresh, intensity=intensity, time_steps=time_steps, dt=dt,
            enable_astrocyte=True, alpha=alpha, k=k)
        QAZ_after_astro = weights_2d_astro * weights_mask_XY
        all_weight_matrices.append(QAZ_after_astro.copy())
        weights_init_XY = torch.Tensor(weights_2d_astro).float()
        all_routes.append(f"Route with astrocytes (cycle {iq+1}): {positions_astro}")
        route_lengths.append(len(positions_astro)-1)
        print(f"\n{'='*60}")
        print(f"CYCLE {iq+1}/20: VERIFICATION WITHOUT ASTROCYTES")
        print(f"{'='*60}")
        positions_no_astro, weights_2d_no_astro = setup_and_run_simulation(
            NA=NA, weights_mask_XY=weights_mask_XY, weights_init_XY=weights_init_XY, weights_init_XI=weights_init_XI,
            n_steps=n_steps, current_position=current_position, goal=goal, learning_rate=learning_rate,
            wmin=wmin, wmax=wmax, weight_decay=weight_decay, post_spike_weight_decay=post_spike_weight_decay,
            reset=reset, refrac=refrac, thresh=thresh, intensity=intensity, time_steps=time_steps, dt=dt,
            enable_astrocyte=False, alpha=alpha, k=k)
        all_routes.append(f"Route without astrocytes (cycle {iq+1}): {positions_no_astro}")
        route_lengths.append(len(positions_no_astro)-1)
    weights_filename = f'{results_dir}/weight_matrices_{experiment_num}.txt'
    with open(weights_filename, 'w') as f:
        f.write(f"WEIGHT MATRICES: EXPERIMENT RESULTS #{experiment_num}\n")
        f.write("=" * 50 + "\n\n")
        for i, weight_matrix in enumerate(all_weight_matrices):
            if i == 0:
                f.write(f"INITIAL WEIGHT MATRIX:\n")
            else:
                f.write(f"WEIGHT MATRIX AFTER CYCLE {i} WITH ASTROCYTES:\n")
            np.savetxt(f, weight_matrix, fmt='%.4f')
            f.write("\n" + "-" * 30 + "\n\n")
    routes_filename = f'{results_dir}/navigation_routes_{experiment_num}.txt'
    with open(routes_filename, 'w') as f:
        f.write(f"NAVIGATION ROUTES: EXPERIMENT RESULTS #{experiment_num}\n")
        f.write("=" * 50 + "\n\n")
        f.write("ALL ROUTES:\n")
        f.write("=" * 30 + "\n")
        for i, route in enumerate(all_routes):
            f.write(f"{i+1:2d}. {route}\n")
        f.write("\n")
        f.write("ROUTE LENGTHS:\n")
        f.write("=" * 30 + "\n")
        for i, length in enumerate(route_lengths):
            f.write(f"{i+1:2d}. Route length: {length}\n")
    weights_mat_filename = f'{results_dir}/weight_matrices_{experiment_num}.mat'
    sio.savemat(weights_mat_filename, {'weight_matrices': np.array(all_weight_matrices)})
    lengths_mat_filename = f'{results_dir}/route_lengths_{experiment_num}.mat'
    sio.savemat(lengths_mat_filename, {'route_lengths': np.array(route_lengths)})

print("START OF EXPERIMENT SERIES")
print("=" * 50)
for i in range(5):
    print(f"\nEXPERIMENT {i+1}/5")
    print("-" * 20)
    run_navigation_experiment(i+1)
print("\n" + "=" * 50)
print("ALL 5 EXPERIMENTS COMPLETED!")

START OF EXPERIMENT SERIES

EXPERIMENT 1/5
--------------------

INITIAL RUN WITHOUT ASTROCYTES


C:\Users\Юля\AppData\Local\Temp\ipykernel_31928\3247724313.py:90: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer("thresh", torch.tensor(thresh, dtype=torch.float))


success

RUN WITH ASTROCYTES 1 OUT OF 20
success

CYCLE 1/20: VERIFICATION WITHOUT ASTROCYTES
success

RUN WITH ASTROCYTES 2 OUT OF 20
success

CYCLE 2/20: VERIFICATION WITHOUT ASTROCYTES
success

RUN WITH ASTROCYTES 3 OUT OF 20
success

CYCLE 3/20: VERIFICATION WITHOUT ASTROCYTES
success

RUN WITH ASTROCYTES 4 OUT OF 20
success

CYCLE 4/20: VERIFICATION WITHOUT ASTROCYTES
success

RUN WITH ASTROCYTES 5 OUT OF 20
success

CYCLE 5/20: VERIFICATION WITHOUT ASTROCYTES
success

RUN WITH ASTROCYTES 6 OUT OF 20
success

CYCLE 6/20: VERIFICATION WITHOUT ASTROCYTES
success

RUN WITH ASTROCYTES 7 OUT OF 20
success

CYCLE 7/20: VERIFICATION WITHOUT ASTROCYTES
success

RUN WITH ASTROCYTES 8 OUT OF 20
success

CYCLE 8/20: VERIFICATION WITHOUT ASTROCYTES
success

RUN WITH ASTROCYTES 9 OUT OF 20
success

CYCLE 9/20: VERIFICATION WITHOUT ASTROCYTES
success

RUN WITH ASTROCYTES 10 OUT OF 20
success

CYCLE 10/20: VERIFICATION WITHOUT ASTROCYTES
success

RUN WITH ASTROCYTES 11 OUT OF 20
success

CYCLE 1